In [99]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [100]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [101]:
target = 'Y'

In [102]:
X = train.drop(columns=[target])
y = train[target]

In [103]:
X['Outlet_Age'] = 2026 - X['X8']
test['Outlet_Age'] = 2026 - test['X8']

X['Price_per_weight'] = X['X6'] / X['X2']
test['Price_per_weight'] = test['X6'] / test['X2']

X['log_X6'] = np.log1p(X['X6'])
test['log_X6'] = np.log1p(test['X6'])

X['mrp_x_age'] = X['X6'] * X['Outlet_Age']
test['mrp_x_age'] = test['X6'] * test['Outlet_Age']

# Fix visibility zeros per item type (keep your good approach)
X['X4'] = X['X4'].replace(0, np.nan)
X['X4'] = X.groupby('X5')['X4'].transform(lambda x: x.fillna(x.mean()))
test['X4'] = test['X4'].replace(0, np.nan)
test['X4'] = test.groupby('X5')['X4'].transform(lambda x: x.fillna(x.mean()))

# Normalize visibility by item type mean (keep your good approach)
X['X4'] = X['X4'] / X.groupby('X5')['X4'].transform('mean')
test['X4'] = test['X4'] / test.groupby('X5')['X4'].transform('mean')


X['X3'] = X['X3'].replace({'LF': 'Low Fat', 'low fat': 'Low Fat', 'reg': 'Regular'})
test['X3'] = test['X3'].replace({'LF': 'Low Fat', 'low fat': 'Low Fat', 'reg': 'Regular'})

In [104]:
X = X.drop(columns=['X1', 'X8'])   # drop X8 (have Outlet_Age), X6 (have log_X6)
test = test.drop(columns=['X1', 'X8'])

In [105]:
cat_cols = list(X.select_dtypes(include=['object']).columns)
num_cols = list(X.select_dtypes(exclude=['object']).columns)

C:\Users\SecureTech\AppData\Local\Temp\ipykernel_6644\1855541669.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = list(X.select_dtypes(include=['object']).columns)


In [106]:
for col in cat_cols:
    X[col] = X[col].fillna('Missing')
    test[col] = test[col].fillna('Missing')

for col in num_cols:
    X[col] = X[col].fillna(X[col].mean())
    test[col] = test[col].fillna(test[col].mean())

In [107]:
for col in cat_cols:
    X[col] = X[col].astype(str)
    test[col] = test[col].astype(str)

In [108]:
for col in cat_cols:
    X_train[col] = X_train[col].astype(str)
    X_val[col] = X_val[col].astype(str)

In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.01,
        depth=6,
        l2_leaf_reg=5,
        min_data_in_leaf=10,
        random_seed=42,
        verbose=False
    )

    model.fit(X_tr, y_tr,
              cat_features=cat_cols,
              eval_set=(X_val, y_val),
              early_stopping_rounds=100)

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(test) / 5

oof_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f'OOF RMSE: {oof_rmse:.4f}')

submission = pd.DataFrame({'row_id': test.index, 'Y': test_preds})
submission.to_csv('sample_submission.csv', index=False)
print(submission.head())

KeyboardInterrupt: 